<a href="https://colab.research.google.com/github/yilinw762/ds2002-fa26/blob/main/notebooks/03-pandas-cleaning/2026_09_16_%E2%80%94_Pandas_Core_Ops_%E2%80%94_Studio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Pandas Core Ops

**Studio — 2026-09-16 · Fall 2026**  
**Class time:** 45 minutes

---

## Two halves today

**Part 1** is a short drill on the copy trap from Monday, because it is the bug that quietly changes your numbers instead of raising an error. Budget about 15 minutes.

**Part 2** walks one deliverable end to end: a per-vendor summary a game-day manager could act on. Three sources, a join that does not behave, and a report at the end. Those builds are **worked examples** — run each one, read what it prints, and stop me when something does not make sense.

The Part 2 data has problems planted in it, and catching them is the actual skill. A report you cannot defend is worth nothing, however good the code looks.

The **checkpoint** at the very end is the part you do yourself.

---

## Part 1 — The copy trap, with the damage visible

Monday you saw `.copy()` on a slide. Here you watch what happens without it, which is the only way it sticks.

Run the next cell for a small frame to experiment on. The real files arrive in Part 2.

In [ ]:
import pandas as pd

print('pandas', pd.__version__)

sales = pd.DataFrame({
    'order_id': [101, 102, 103, 104, 105],
    'item': ['Rain Poncho', 'Cheeseburger', 'Rain Poncho', 'Hot Dog', 'Foam Finger'],
    'qty': [5, 2, 8, 3, 1],
    'price': [6.00, 7.50, 6.00, 4.50, 12.00],
})
sales

### Bad result 1 — the edit that goes nowhere

The ponchos went on sale at $4.50. This is how nearly everybody writes it the first time. Run it, and read the two printed prices before you read any warning.

In [ ]:
print('before:', sales.loc[sales['item'] == 'Rain Poncho', 'price'].tolist())

sales[sales['item'] == 'Rain Poncho']['price'] = 4.50

print('after: ', sales.loc[sales['item'] == 'Rain Poncho', 'price'].tolist())

Nothing changed, and nothing crashed.

That line is two operations, not one. `sales[sales['item'] == 'Rain Poncho']` builds a brand-new temporary frame holding the two poncho rows. `['price'] = 4.50` then sets the price **on that temporary**. Nothing was holding a reference to it, so Python discarded it a microsecond later. Your data never saw the change.

This is the expensive kind of bug. The code looks right, it does not raise, and the number you report is the old one.

### Bad result 2 — the edit that lands somewhere you did not mean

Now the version where the slice goes into a variable first.

In [ ]:
rain = sales[sales['item'] == 'Rain Poncho']
rain['sale_price'] = 4.50

print(rain)
print()
print("'sale_price' in rain? ", 'sale_price' in rain.columns)
print("'sale_price' in sales?", 'sale_price' in sales.columns)

This one did something — `rain` has the new column. What it did not do is touch `sales`.

Whether you also get a warning here depends on your pandas version, which is exactly why the printed columns matter more than the warnings:

- On **pandas 2.x** you get `SettingWithCopyWarning`, because older pandas could not promise whether `rain` shared memory with `sales`.
- On **pandas 3.x** you get nothing at all. Copy-on-Write is always on, so `rain` is guaranteed to be its own frame.

The warning was never the lesson. The lesson is that you have to know which frame your assignment lands in, and slicing never hands you the original.

### The fix — two tools, for two different jobs

Decide what you want before you type. There are exactly two answers.

In [ ]:
# Tool 1 -- you want a separate frame to work on. Say so with .copy().
rain = sales[sales['item'] == 'Rain Poncho'].copy()
rain['sale_price'] = 4.50
print('rain has sale_price   :', 'sale_price' in rain.columns)
print('sales left alone      :', 'sale_price' not in sales.columns)
print()

# Tool 2 -- you want to change the original. One operation, so it lands.
sales.loc[sales['item'] == 'Rain Poncho', 'price'] = 4.50
print('sales poncho price now:',
      sales.loc[sales['item'] == 'Rain Poncho', 'price'].tolist())

Keep this table until it is reflex:

| What you want | What to write |
|---|---|
| A separate frame you can modify freely | `sub = df[mask].copy()` |
| To change the original in place | `df.loc[mask, 'col'] = value` |
| Nothing, ever | `df[mask]['col'] = value` |

The third row is not a style preference. It does not work.

### Your turn 1 — make the change actually land

The Hot Dog price should be `5.00` in `sales` itself. The broken version is sitting in the cell as a comment; do not use it. Write the version that works, then print the price to prove it.

In [ ]:
# Broken -- leave it commented, it silently does nothing:
#     sales[sales['item'] == 'Hot Dog']['price'] = 5.00

# TODO: the version that changes `sales`

# TODO: print the Hot Dog price -- expect [5.0]

### Your turn 2 — a working copy that leaves the original alone

Build `bulk`: only the rows with `qty >= 3`, plus a new `line_total` column equal to `qty * price`. `sales` must come out of this unchanged.

Two things must be true when you are done: `bulk` has a `line_total` column, and `sales` does not.

In [ ]:
# TODO: bulk = ...
# TODO: bulk['line_total'] = ...

# Uncomment these to check yourself:
# assert 'line_total' in bulk.columns
# assert 'line_total' not in sales.columns
# print(bulk)

---

## Part 2 — The vendor report

Three sources: order lines, a vendor roster, and a revenue target per zone. Run the next cell to load them.

In [ ]:
import pandas as pd
from io import StringIO

orders = pd.read_csv(StringIO('''order_id,vendor_id,item,qty,price
1,V-01,Cheeseburger,2,7.50
2,V-10,Foam Finger,1,12.00
3,V-01,Hot Dog,3,4.50
4,V-18,Rain Poncho,5,6.00
5,V-10,UVA T-Shirt,1,24.00
6,V-05,Chicken Tacos,4,6.50
7,V-18,Rain Poncho,8,6.00
8,V-42,Kettle Corn,3,5.00'''))

vendors = pd.read_csv(StringIO('''vendor_id,vendor_name,zone
V-01,Hoos Burgers,A
V-05,Rotunda Tacos,B
V-10,Cav Merch North,A
V-18,Rally Rain Gear,C
V-18,Rally Rain Gear,C'''))

targets = pd.read_csv(StringIO('''zone,revenue_target
A,80
B,25
C,60'''))

print('orders:', orders.shape, '| vendors:', vendors.shape, '| targets:', targets.shape)
orders

### Worked example — the merge, done carefully

Here is one merge done properly, so the pattern is on the screen before you write anything. Three things happen: record the baseline, merge with `indicator=True`, then compare against the baseline.

In [ ]:
baseline_rows = len(orders)
orders['revenue'] = orders['qty'] * orders['price']
baseline_revenue = orders['revenue'].sum()
print(f'before: {baseline_rows} rows, ${baseline_revenue:.2f}')

check = orders.merge(vendors, on='vendor_id', how='left', indicator=True)
print(f'after:  {len(check)} rows, ${check["revenue"].sum():.2f}')
print()
print(check['_merge'].value_counts())

Eight orders went in and **ten** came out, and the revenue total jumped by $78. Both symptoms point at the same cause, and it is in the `vendors` table, not in the orders.

Note that `_merge` says `both = 9` and `left_only = 1`. Nine matches out of eight orders is already impossible, which is the tell.

Find it before you go further — everything downstream inherits this bug.

In [ ]:
# Which vendor_id appears more than once in the vendor list?
print(vendors['vendor_id'].value_counts())
print()
print('duplicated vendor rows:', vendors.duplicated().sum())

### Build 1 — fix the vendor list, then merge

Drop the duplicate vendor row, then join the roster onto `orders` with an indicator.

The two `assert` lines are the point of this cell. A merge that silently changes your row count or your revenue total is the bug we just watched happen, and an assert turns it from something you might notice into something that stops the notebook.

In [ ]:
clean_vendors = vendors.drop_duplicates()
assert clean_vendors['vendor_id'].is_unique, 'vendor_id must be unique to join safely'

joined = orders.merge(clean_vendors, on='vendor_id', how='left', indicator=True)

print('rows:', len(joined), '(expected', baseline_rows, ')')
print('revenue:', joined['revenue'].sum(), '(expected', baseline_revenue, ')')
assert len(joined) == baseline_rows
assert joined['revenue'].sum() == baseline_revenue

### Build 2 — handle the vendor nobody has heard of

One order belongs to a vendor that is not on the roster. Three options, and it is a judgment call:

1. Drop it — clean report, understated revenue.
2. Keep it with a blank name — the revenue total stays right, but it shows up as `NaN` in every chart and table.
3. Label it `'Unknown vendor'` and keep it in a zone called `'Unassigned'`.

We take option 3 below. Notice that the first thing the cell does is print what the decision is worth, because the size of the number is what makes it a decision.

In [ ]:
missing = joined[joined['_merge'] == 'left_only']
print(f'unmatched orders: {len(missing)} | '
      f'revenue at stake: ${missing["revenue"].sum():.2f}')

joined['vendor_name'] = joined['vendor_name'].fillna('Unknown vendor')
joined['zone'] = joined['zone'].fillna('Unassigned')
joined = joined.drop(columns='_merge')
joined[['order_id', 'vendor_id', 'vendor_name', 'zone', 'revenue']]

**Why option 3:** $15 is 8% of the night's $183.50, and a row labeled "Unknown vendor" is a question somebody can go answer. A dropped row is a question nobody knows to ask.

All three choices are defensible as long as you state the tradeoff and report the amount. The checkpoint asks for *your* call, so it does not have to be this one.

### Build 3 — the per-vendor summary

One row per vendor:

- `orders` — how many orders
- `units` — total quantity
- `revenue` — total revenue
- `avg_ticket` — average revenue per order, rounded to 2 decimals

Sorted by revenue, highest first. `.agg()` with named outputs is what gets the column names above instead of a stack of `sum` and `mean` headers.

In [ ]:
summary = (joined.groupby('vendor_name')
           .agg(orders=('order_id', 'count'),
                units=('qty', 'sum'),
                revenue=('revenue', 'sum'),
                avg_ticket=('revenue', 'mean'))
           .round(2)
           .sort_values('revenue', ascending=False))
summary

### Build 4 — did each zone hit its target?

Total revenue by zone, `targets` joined on, and a `hit_target` boolean. Then a one-line sentence per zone that a manager could actually read.

The `pd.isna` branch matters: the `Unassigned` zone we invented in Build 2 has no target, and without that check it would silently report as a miss.

In [ ]:
by_zone = joined.groupby('zone', as_index=False)['revenue'].sum()
scored = by_zone.merge(targets, on='zone', how='left')
scored['hit_target'] = scored['revenue'] >= scored['revenue_target']

for row in scored.itertuples():
    if pd.isna(row.revenue_target):
        print(f'Zone {row.zone}: ${row.revenue:.2f}, no target set.')
    else:
        verdict = 'hit' if row.hit_target else 'missed'
        print(f'Zone {row.zone}: ${row.revenue:.2f} against a '
              f'${row.revenue_target:.2f} target -- {verdict}.')

### Build 5 — the one number that matters

Rain gear is the thing we can act on. Total poncho units sold, and what share of overall revenue they represent.

In [ ]:
ponchos = joined[joined['item'] == 'Rain Poncho']
share = 100 * ponchos['revenue'].sum() / joined['revenue'].sum()
print(f"{int(ponchos['qty'].sum())} ponchos sold, "
      f"{share:.1f}% of the night's revenue")

That last number is the whole report in one sentence: ponchos are 13 units and 42.5% of the night's revenue, on a rainy game day. Everything above it exists to make that sentence trustworthy.

---

## Checkpoint (participation)

Report the one line that fixes the copy trap, your row count and revenue total after the merge, and what you decided to do with the unknown vendor.

Work in groups if you wish, then fill in the cell below **yourself**. Paste the printed output (or a screenshot of it) into this week's **Studio Checkpoint** in Canvas by **Thursday 11:59pm ET**. One submission per person, not per group.

In [ ]:
# Checkpoint
copy_fix = 'TODO'          # TODO: the line you wrote to change `sales` itself
rows_after_merge = None    # TODO: should equal 8
revenue_after_merge = None # TODO: should equal the baseline
unknown_vendor_call = 'TODO'  # TODO: what you did with V-42, and why
top_vendor = 'TODO'        # TODO: highest-revenue vendor from your Build 3 summary

print('copy fix:', copy_fix)
print('rows after merge:', rows_after_merge)
print('revenue after merge:', revenue_after_merge)
print('unknown vendor:', unknown_vendor_call)
print('top vendor:', top_vendor)